In [1]:
import os
import cv2
import torch
import torchaudio
from torchaudio.io import StreamReader
from tqdm import tqdm
import subprocess

In [2]:
DATA_DIR  = "DATA"
VIDEO_DIR = os.path.join(DATA_DIR, "raw_videos")
IMAGE_DIR = os.path.join(DATA_DIR, "images")
AUDIO_DIR = os.path.join(DATA_DIR, "audio")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

assert os.path.exists(VIDEO_DIR), "raw_videos folder not found"

VIDEO_DIR, IMAGE_DIR, AUDIO_DIR


('DATA\\raw_videos', 'DATA\\images', 'DATA\\audio')

In [3]:
DATA_DIR  = "DATA"
VIDEO_DIR = os.path.join(DATA_DIR, "raw_videos")
IMAGE_DIR = os.path.join(DATA_DIR, "images")
AUDIO_DIR = os.path.join(DATA_DIR, "audio")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

assert os.path.exists(VIDEO_DIR), "raw_videos folder not found"

VIDEO_DIR, IMAGE_DIR, AUDIO_DIR


('DATA\\raw_videos', 'DATA\\images', 'DATA\\audio')

In [4]:
def extract_audio_ffmpeg(video_path, audio_out):
    cmd = [
        "ffmpeg",
        "-y",                    # overwrite if exists
        "-i", video_path,        # input video
        "-ac", "1",              # mono
        "-ar", "16000",          # 16 kHz
        "-vn",                   # no video
        audio_out
    ]

    subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )


In [5]:
def extract_frame_and_audio(video_path, image_out, audio_out):
    # ---- extract middle frame ----
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
        ret, frame = cap.read()
        if ret:
            cv2.imwrite(image_out, frame)

    cap.release()

    # ---- extract audio using FFmpeg ----
    extract_audio_ffmpeg(video_path, audio_out)


In [6]:
processed = 0
skipped = 0

for root, _, files in tqdm(list(os.walk(VIDEO_DIR))):
    for f in files:

        if not f.lower().endswith(".mp4"):
            continue

        parts = f.replace(".mp4", "").split("-")

        # strict RAVDESS filename check
        if len(parts) != 7:
            skipped += 1
            continue

        # keep only full audio-video samples (01)
        if parts[0] != "01":
            skipped += 1
            continue

        base = f.replace(".mp4", "")
        img_path = os.path.join(IMAGE_DIR, base + ".jpg")
        wav_path = os.path.join(AUDIO_DIR, base + ".wav")

        if os.path.exists(img_path) and os.path.exists(wav_path):
            skipped += 1
            continue

        extract_frame_and_audio(
            os.path.join(root, f),
            img_path,
            wav_path
        )

        processed += 1

processed, skipped


100%|█████████████████████████████████████████████████████████████████████████████████| 26/26 [00:00<00:00, 214.62it/s]


(0, 1441)

In [7]:
len(os.listdir(IMAGE_DIR)), len(os.listdir(AUDIO_DIR))


(1440, 1440)